# ResUNet Comparison Notebook

Train a ResUNet on the same segmentation data and with the same hyperparameters as the current U-Net baseline, then record test performance for side-by-side comparison.


In [ ]:
# Colab setup: mount Drive, expose project root, and install dependencies.
# Run this cell once per new Colab runtime before importing project code.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Set to True if you see "Transport endpoint is not connected" from Drive paths.
FORCE_REMOUNT_DRIVE = False


def _safe_cwd() -> tuple[Path, bool]:
    try:
        return Path.cwd(), False
    except OSError as e:
        if getattr(e, 'errno', None) == 107:
            # The current working directory points to a broken mount.
            os.chdir('/')
            return Path('/'), True
        raise


def _exists_safe(path: Path) -> bool:
    try:
        return path.exists()
    except OSError as e:
        if getattr(e, 'errno', None) == 107:
            return False
        raise


cwd_path, cwd_was_broken = _safe_cwd()
if cwd_was_broken:
    FORCE_REMOUNT_DRIVE = True
    print('Detected disconnected current directory; switched cwd to /.')
    print('Auto-enabling force remount for this run.')

IS_COLAB = importlib.util.find_spec('google.colab') is not None
if IS_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=FORCE_REMOUNT_DRIVE)
    print('Google Drive mounted at /content/drive')
else:
    print('Not running in Colab; skipping Google Drive mount.')

project_candidates = [
    Path('/content/drive/MyDrive/comp4471Project'),
    Path('/content/drive/MyDrive/Comp4471Project'),
    Path('/content/drive/MyDrive/COMP4471Project'),
    Path('/Users/chyo/My Drive/comp4471Project'),
    cwd_path,
]

PROJECT_ROOT = next((p for p in project_candidates if _exists_safe(p / 'src')), None)

drive_root = Path('/content/drive/MyDrive')
if PROJECT_ROOT is None and _exists_safe(drive_root):
    # Shallow fallback search for a folder that contains src/preprocessing.py.
    for p in drive_root.glob('*'):
        if _exists_safe(p / 'src' / 'preprocessing.py'):
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find project root containing src/. '
        'Put the full comp4471Project folder in Google Drive, then rerun this cell.'
    )

PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Added project root to sys.path and set working directory.')


def _missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def _pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


packages_to_install = []
if _missing('medpy'):
    packages_to_install.append('medpy')
if _missing('albumentations'):
    packages_to_install.append('albumentations')
if _missing('cv2'):
    packages_to_install.append('opencv-python-headless')

if packages_to_install:
    print('Installing:', ', '.join(packages_to_install))
    _pip_install(packages_to_install)
    print('Install complete.')
else:
    print('All required libraries are already installed.')


In [ ]:
import importlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import src.preprocessing as _preprocessing
import src.segmentation as _segmentation

# Reload project modules so Colab picks up latest edits from Drive.
importlib.reload(_preprocessing)
importlib.reload(_segmentation)

from src.preprocessing import (
    discover_image_mask_pairs,
    discover_images,
    generate_pseudo_masks_for_split,
    stratified_split,
    stratified_split_pairs,
    SegmentationDataset,
)
from src.segmentation import compute_segmentation_metrics
from src.utils import AverageMeter, EarlyStopping, fmt_time

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
# Find the same segmentation data used by the U-Net pipeline.
# Update DATA_DIR if your dataset lives somewhere else.
import inspect
from collections import deque

DATA_DIR = None
SPLIT_RATIOS = (0.8, 0.1, 0.1)


def _iter_dirs_bfs(base: Path, max_depth: int = 2):
    if not base.exists():
        return
    q = deque([(base, 0)])
    seen = set()
    while q:
        cur, depth = q.popleft()
        try:
            key = str(cur.resolve())
        except Exception:
            key = str(cur)
        if key in seen:
            continue
        seen.add(key)
        yield cur
        if depth >= max_depth:
            continue
        try:
            children = [d for d in cur.iterdir() if d.is_dir()]
        except Exception:
            continue
        for child in children:
            q.append((child, depth + 1))


explicit_candidates = [
    Path(DATA_DIR) if DATA_DIR else None,
    PROJECT_ROOT / 'data',
    PROJECT_ROOT / 'dataset',
    Path('/content/drive/MyDrive/comp4471Project/data'),
    Path('/content/drive/MyDrive/comp4471Project/dataset'),
]

candidate_roots = [p for p in explicit_candidates if p is not None and p.exists()]

# Explore nearby folders under project root.
candidate_roots.extend(list(_iter_dirs_bfs(PROJECT_ROOT, max_depth=2)))

# Add likely dataset folders directly under MyDrive (keyword-filtered).
drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    keywords = ('dataset', 'data', 'tumor', 'brain', 'mri', 'kaggle', 'archive', 'comp4471')
    for top in drive_root.iterdir():
        if top.is_dir() and any(k in top.name.lower() for k in keywords):
            candidate_roots.append(top)
            candidate_roots.extend(list(_iter_dirs_bfs(top, max_depth=1)))

# De-duplicate roots while preserving order.
unique_roots = []
seen = set()
for p in candidate_roots:
    try:
        key = str(p.resolve())
    except Exception:
        key = str(p)
    if key in seen:
        continue
    seen.add(key)
    unique_roots.append(p)

if not unique_roots:
    raise FileNotFoundError('Could not find candidate data roots. Set DATA_DIR manually.')

pseudo_root = PROJECT_ROOT / 'outputs' / 'pseudo_masks_resunet'
kwargs = {}
sig = inspect.signature(discover_image_mask_pairs)
if 'allow_pseudo_fallback' in sig.parameters:
    kwargs['allow_pseudo_fallback'] = True
if 'pseudo_output_dir' in sig.parameters:
    kwargs['pseudo_output_dir'] = str(pseudo_root)

pairs = None
data_root = None
for cand in unique_roots:
    try:
        found = discover_image_mask_pairs(str(cand), **kwargs)
        if found:
            pairs = found
            data_root = cand
            break
    except FileNotFoundError:
        continue

if pairs is None:
    checked_preview = '\n'.join(f'  - {p}' for p in unique_roots[:25])
    raise FileNotFoundError(
        'Could not locate a usable dataset root automatically.\n'
        'Set DATA_DIR to your actual dataset directory and rerun.\n'
        f'Checked roots (first 25):\n{checked_preview}'
    )

print('Data root:', data_root)
print('Found image-mask pairs:', len(pairs))

train_pairs, val_pairs, test_pairs = stratified_split_pairs(
    pairs, ratios=SPLIT_RATIOS, seed=SEED
)
print(f'Train={len(train_pairs)} Val={len(val_pairs)} Test={len(test_pairs)}')


In [ ]:
# Same hyperparameters as the current U-Net baseline
IMG_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 130
LR = 3e-4
PATIENCE = 12
BASE_FILTERS = 64
SAVE_DIR = Path('models/segmentation')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = SAVE_DIR / 'best_resunet.pth'
RESULTS_DIR = Path('outputs/metrics')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / 'resunet_comparison.json'
U_NET_RESULTS_PATH = RESULTS_DIR / 'unet_metrics.json'
COMPARISON_PATH = RESULTS_DIR / 'unet_vs_resunet_comparison.csv'

train_ds = SegmentationDataset(train_pairs, img_size=IMG_SIZE, augment=True)
val_ds = SegmentationDataset(val_pairs, img_size=IMG_SIZE, augment=False)
test_ds = SegmentationDataset(test_pairs, img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print('Batches:', len(train_loader), len(val_loader), len(test_loader))


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = F.relu(out + identity, inplace=True)
        return out

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch, stride=2)

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = ResidualBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        x = torch.cat([skip, x], dim=1)
        return self.block(x)

class ResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        f = base_filters
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, f, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(f),
            nn.ReLU(inplace=True),
        )
        self.enc1 = ResidualBlock(f, f)
        self.enc2 = DownBlock(f, f * 2)
        self.enc3 = DownBlock(f * 2, f * 4)
        self.enc4 = DownBlock(f * 4, f * 8)
        self.enc5 = DownBlock(f * 8, f * 16)
        self.dec4 = UpBlock(f * 16, f * 8, f * 8)
        self.dec3 = UpBlock(f * 8, f * 4, f * 4)
        self.dec2 = UpBlock(f * 4, f * 2, f * 2)
        self.dec1 = UpBlock(f * 2, f, f)
        self.outc = nn.Conv2d(f, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.stem(x)
        x2 = self.enc1(x1)
        x3 = self.enc2(x2)
        x4 = self.enc3(x3)
        x5 = self.enc4(x4)
        x6 = self.enc5(x5)
        x = self.dec4(x6, x5)
        x = self.dec3(x, x4)
        x = self.dec2(x, x3)
        x = self.dec1(x, x2)
        return self.outc(x)

def dice_bce_loss(logits, targets, bce_weight=0.5, smooth=1.0):
    bce = nn.BCEWithLogitsLoss()(logits, targets)
    probs = torch.sigmoid(logits).view(-1)
    t = targets.view(-1)
    inter = (probs * t).sum()
    dice = 1.0 - (2.0 * inter + smooth) / (probs.sum() + t.sum() + smooth)
    return bce_weight * bce + (1.0 - bce_weight) * dice

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_meter = AverageMeter()
    all_metrics = []
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        loss = dice_bce_loss(logits, masks)
        loss_meter.update(loss.item(), imgs.size(0))
        preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()
        gt = masks.cpu().numpy()
        for i in range(preds.shape[0]):
            m = compute_segmentation_metrics(preds[i].squeeze(), gt[i].squeeze())

            # Backward compatibility with older metric dicts.
            if 'iou' not in m:
                pred_bin = preds[i].squeeze().astype(bool)
                gt_bin = gt[i].squeeze().astype(bool)
                inter = np.logical_and(pred_bin, gt_bin).sum()
                union = np.logical_or(pred_bin, gt_bin).sum()
                m['iou'] = float(inter / (union + 1e-8))
            if 'hd95' not in m:
                m['hd95'] = float(m.get('hausdorff95', 0.0))

            all_metrics.append(m)

    if all_metrics:
        avg = {
            k: float(np.mean([m[k] for m in all_metrics]))
            for k in all_metrics[0]
            if k not in ('hd95', 'hausdorff95')
        }
        hd95_values = np.asarray(
            [m.get('hd95', m.get('hausdorff95', np.inf)) for m in all_metrics],
            dtype=np.float64,
        )
        finite_hd95 = hd95_values[np.isfinite(hd95_values)]
        invalid_count = int((~np.isfinite(hd95_values)).sum())
        hd95_mean = float(finite_hd95.mean()) if finite_hd95.size else float('inf')
        avg['hd95'] = hd95_mean
        avg['hausdorff95'] = hd95_mean
        avg['hd95_invalid_count'] = invalid_count
        avg['hd95_invalid_rate'] = float(invalid_count / len(hd95_values))
    else:
        avg = {}

    avg.setdefault('iou', 0.0)
    avg.setdefault('hd95', 0.0)
    avg.setdefault('hausdorff95', avg['hd95'])
    avg.setdefault('hd95_invalid_count', 0)
    avg.setdefault('hd95_invalid_rate', 0.0)
    return float(loss_meter.avg), avg


def _load_baseline_metrics():
    if U_NET_RESULTS_PATH.exists():
        try:
            baseline = json.loads(U_NET_RESULTS_PATH.read_text())
        except Exception:
            baseline = {}
    else:
        baseline = {}

    if 'test_iou' not in baseline:
        baseline['test_iou'] = baseline.get('iou', 0.0)
    if 'test_hd95' not in baseline:
        baseline['test_hd95'] = baseline.get(
            'hd95',
            baseline.get('hausdorff95', baseline.get('test_hausdorff95', 0.0)),
        )
    baseline['model'] = baseline.get('model', 'U-Net')
    return baseline

def train_resunet(model, train_loader, val_loader):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    early_stop = EarlyStopping(patience=PATIENCE)
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_dice': [],
        'val_iou': [],
        'val_hd95': [],
        'val_hd95_invalid_rate': [],
        'lr': [],
    }
    best_dice = 0.0
    train_start = __import__('time').time()
    print(f"{'=' * 70}")
    print(f"  Training ResUNet  |  {EPOCHS} epochs  |  patience={PATIENCE}  |  device={DEVICE}")
    print(f"  Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")
    print(f"{'=' * 70}")
    for epoch in range(1, EPOCHS + 1):
        epoch_start = __import__('time').time()
        model.train()
        meter = AverageMeter()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = dice_bce_loss(logits, masks)
            loss.backward()
            optimizer.step()
            meter.update(loss.item(), imgs.size(0))
        train_loss = meter.avg
        val_loss, val_metrics = evaluate(model, val_loader)
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_metrics.get('dice', 0.0))
        history['val_iou'].append(val_metrics.get('iou', 0.0))
        history['val_hd95'].append(val_metrics.get('hd95', 0.0))
        history['val_hd95_invalid_rate'].append(val_metrics.get('hd95_invalid_rate', 0.0))
        history['lr'].append(optimizer.param_groups[0]['lr'])
        epoch_time = __import__('time').time() - epoch_start
        eta = epoch_time * (EPOCHS - epoch)
        print(f"Epoch {epoch:03d}/{EPOCHS} | loss: {train_loss:.4f}/{val_loss:.4f} | dice={val_metrics.get('dice', 0.0):.4f} | iou={val_metrics.get('iou', 0.0):.4f} | hd95={val_metrics.get('hd95', 0.0):.4f} | lr={optimizer.param_groups[0]['lr']:.2e} | {fmt_time(epoch_time)}/ep ETA {fmt_time(eta)}")
        if val_metrics.get('dice', 0.0) > best_dice:
            best_dice = val_metrics['dice']
            torch.save(model.state_dict(), SAVE_PATH)
            print(f'  -> Best model saved (Dice={best_dice:.4f})')
        if early_stop(val_loss):
            print(f'Early stopping at epoch {epoch}')
            break
    total_time = __import__('time').time() - train_start
    print(f"{'=' * 70}")
    print(f"  Training finished in {fmt_time(total_time)}  |  Best Dice={best_dice:.4f}")
    print(f"{'=' * 70}")
    return history


In [ ]:
model = ResUNet(in_channels=1, out_channels=1, base_filters=BASE_FILTERS).to(DEVICE)
print('ResUNet parameters:', sum(p.numel() for p in model.parameters()))


def _path_exists_safe(path: Path) -> bool:
    try:
        return path.exists()
    except OSError as e:
        if getattr(e, 'errno', None) == 107:
            raise RuntimeError(
                f'Filesystem endpoint disconnected while checking {path}. '
                'Rerun the setup/mount cell (use force_remount=True), then rerun this cell.'
            ) from e
        raise


def _load_state_dict_safe(path: Path):
    try:
        return torch.load(path, map_location=DEVICE)
    except OSError as e:
        if getattr(e, 'errno', None) == 107:
            raise OSError(
                f'Could not read {path} because the filesystem endpoint is disconnected. '
                'Rerun the setup/mount cell to reconnect Google Drive.'
            ) from e
        raise


if _path_exists_safe(SAVE_PATH):
    print('Loading existing ResUNet weights:', SAVE_PATH)
    model.load_state_dict(_load_state_dict_safe(SAVE_PATH))
    history = None
else:
    history = train_resunet(model, train_loader, val_loader)
    if not _path_exists_safe(SAVE_PATH):
        raise FileNotFoundError(
            f'Expected trained weights at {SAVE_PATH}, but the file is unavailable.'
        )
    model.load_state_dict(_load_state_dict_safe(SAVE_PATH))

test_loss, test_metrics = evaluate(model, test_loader)
print('Test loss:', test_loss)
print('Test metrics:', test_metrics)
baseline = _load_baseline_metrics()

results = {
    'model': 'ResUNet',
    'epochs': EPOCHS,
    'lr': LR,
    'patience': PATIENCE,
    'base_filters': BASE_FILTERS,
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'test_loss': test_loss,
    **{f'test_{k}': v for k, v in test_metrics.items()},
}
RESULTS_PATH.write_text(json.dumps(results, indent=2))
comparison_rows = [
    {
        'model': baseline.get('model', 'U-Net'),
        'test_dice': baseline.get('test_dice', baseline.get('dice', 0.0)),
        'test_iou': baseline.get('test_iou', baseline.get('iou', 0.0)),
        'test_hd95': baseline.get(
            'test_hd95',
            baseline.get('hd95', baseline.get('hausdorff95', baseline.get('test_hausdorff95', 0.0))),
        ),
        'test_loss': baseline.get('test_loss', 0.0),
    },
    {
        'model': 'ResUNet',
        'test_dice': test_metrics.get('dice', 0.0),
        'test_iou': test_metrics.get('iou', 0.0),
        'test_hd95': test_metrics.get('hd95', test_metrics.get('hausdorff95', 0.0)),
        'test_loss': test_loss,
    },
]
pd.DataFrame(comparison_rows).to_csv(COMPARISON_PATH, index=False)
print('Saved results to', RESULTS_PATH)
print('Saved comparison table to', COMPARISON_PATH)


## Notes

Use the saved `resunet_comparison.json` file to compare against the U-Net metrics from your current notebook.


## Side-by-side comparison

This notebook now records test IoU and HD95 for both the U-Net baseline and ResUNet, then writes a CSV comparison table to `outputs/metrics/unet_vs_resunet_comparison.csv`.
